In [ ]:
import polars as pl
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import colorir as cl
from plotly.subplots import make_subplots
from analyses import io

In [ ]:
celldf = pl.concat(io.read_celldfs("../runs/pseudopodia/")).with_columns(
    replica=pl.col("replica").cast(pl.UInt32),
    gamma=20 - pl.col("energy").str.split("-").list.get(-1).cast(pl.UInt32),
    displ=(pl.col("center_x") ** 2 + pl.col("center_y") ** 2) ** 0.5
).with_columns(
    mean_displ=pl.col("displ").mean().over("gamma", "replica", "time")
).drop("energy")
celldf

In [ ]:
max_time = celldf["time"].max()
min_steady_time = 3e6
stdf = celldf\
    .group_by(["gamma", "replica"])\
    .agg(
        stime=pl.col("time")\
            .filter(
                pl.col("mean_displ") <= pl.col("mean_displ").filter(pl.col("time") > min_steady_time).mean()
            )\
            .min(),
        ptime=pl.col("time")\
            .filter(
                pl.col("mean_displ") <= 180
            )\
            .min().fill_null(max_time)
    )
stdf

In [ ]:
kactdf = celldf.join(
    stdf, 
    on=["replica", "gamma"]
).filter(
    pl.col("time") > 5e5,
    pl.col("time") < pl.col("stime"),
    pl.col("med_neighbor") == True,
).with_columns(
    mean_x=pl.col("center_x").mean().over("gamma", "replica", "time"),
    mean_y=pl.col("center_y").mean().over("gamma", "replica", "time")
).with_columns(
    dx=pl.col("kact_center_x") - pl.col("center_x"),
    dy=pl.col("kact_center_y") - pl.col("center_y"),
    cell_dx=pl.col("center_x") - pl.col("mean_x"),
    cell_dy=pl.col("center_y") - pl.col("mean_y"),
).with_columns(
    angle=(pl.arctan2(pl.col("dy"), pl.col("dx")).degrees() + 135) % 360,
    cell_angle=(pl.arctan2(pl.col("cell_dy"), pl.col("cell_dx")).degrees() + 135) % 360,
    mag=(pl.col("dx") ** 2 + pl.col("dy") ** 2) ** 0.5,
    cell_mag=(pl.col("cell_dx") ** 2 + pl.col("cell_dy") ** 2) ** 0.5,
).with_columns(
    angle_bin=pl.col("cell_angle").cut(np.linspace(0, 360, 20), include_breaks=True).struct.field("breakpoint"),
    cos_sim=(pl.col("angle") - pl.col("cell_angle")).radians().cos(),
    front=(pl.col("cell_angle") < 30) | (pl.col("cell_angle") >= 330),
    back=(pl.col("cell_angle") > 150) & (pl.col("cell_angle") <= 210),
).with_columns(
    force=pl.col("cos_sim"),
    ux=pl.col("dx") / pl.col("mag"),
    uy=pl.col("dy") / pl.col("mag"),
    cell_ux=pl.col("cell_dx") / pl.col("mag"),
    cell_uy=pl.col("cell_dy") / pl.col("mag")
).drop_nans()  # When dx = 0 and dy = 0 there cant be an angle
kactdf

In [ ]:
angledf = kactdf.group_by("gamma", "angle_bin").agg(
    ux=pl.col("ux").sum(),
    uy=pl.col("uy").sum(),    
    cell_ux=pl.col("cell_ux").sum(),
    cell_uy=pl.col("cell_uy").sum(),    
).with_columns(
    cos_sim=(pl.arctan2("uy", "ux") - pl.arctan2("cell_uy", "cell_ux")).cos()
).sort(
    "gamma"
)
angledf

In [ ]:
px.strip(
    angledf,
    x="gamma",
    y="cos_sim"
).update_traces(
    jitter=1
).update_layout(
    width=500,
    height=300,
    template="plotly_white"
)

In [ ]:
colors = cl.StackPalette.load("darkspectral", palettes_dir="../palettes/")
colors

In [ ]:
for (gamma,), filterdf in kactdf.sort("gamma").group_by("gamma", maintain_order=True):
    filterdf = kactdf.filter(
        pl.col("gamma") == gamma
    )
    bindf = filterdf.with_columns(
        angle_bin=pl.col("cell_angle").cut(np.linspace(0, 360, 20), include_breaks=True).struct.field("breakpoint")
    ).group_by("angle_bin").mean().with_columns(
        cos_sim=(pl.arctan2(pl.col("uy"), pl.col("ux")) - pl.arctan2(pl.col("cell_uy"), pl.col("cell_ux"))).cos()
    ).with_columns(
        strength=pl.col("tot_kact") * pl.col("")
    )
    fig = px.bar_polar(
        bindf,
        r="cos_sim",
        theta="angle_bin",
        direction="counterclockwise"
    ).update_traces(
        marker_color=colors[0]
    )
    fig.update_layout(
        width=300,
        height=300,
        template="plotly_white",
    )
    print(gamma)
    fig.show()

In [ ]:
df = celldf.join(
    stdf, 
    on=["replica", "gamma"]
).filter(
    pl.col("time") < pl.col("stime"),
    pl.col("med_neighbor") == True,
    gamma=12,
    replica=0,
    index=0
)
px.scatter(
    df,
    x="center_x",
    y="center_y"
).add_traces(px.scatter(
    df,
    x="kact_center_x",
    y="kact_center_y",
).update_traces(marker_color="red").data)